In [3]:
require(data.table)
require(tidyverse)
require(dplyr)
require(phyloseq)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)
options(repr.plot.width=20, repr.plot.height=15)

# dada2 leave out run1

## condolidate dada2 runs w/o run1
- is Datee_16S still significant?

-  sequence run 1, selected for the 400 band, 
- migght be that some samples were sequenced at greater depth bc there were less conc of others 


In [4]:
track2=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run2_Seqnums_16S.csv")

track3=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run3_Seqnums_16S.csv")

track4=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run4_Seqnums_16S.csv")

track5=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run5_Seqnums_16S.csv")

In [5]:
track_no1 <- bind_rows(track2, track3, track4, track5)

In [6]:
meta=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/meta_PCR.csv", header=TRUE)

## merging track and meta to run anova for what is driving reads

In [7]:
setnames(track_no1, c("V1"), c("Tubelabel_species"), skip_absent=TRUE)
# Step 1: Replace '-' with '_' in asv_clean$Tube
track_no1$Tubelabel_species <- gsub("-", "_", track_no1$Tubelabel_species)
head(track_no1)

Tubelabel_species,input,filtered,denoisedF,denoisedR,merged,nonchim
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
012024_BEL_CBC_T1_559_MCAV,1377610,1219878,1217341,1217659,1205749,206754
012024_BEL_CBC_T2_591_PAST,975642,854021,822591,819017,756352,487061
012024_BEL_CBC_T2_595_PAST,1186132,1043059,1040328,1040382,1029411,508641
012024_BEL_CBC_T2_599_MCAV,1198173,1059557,1057873,1057921,1051080,401576
012024_BEL_CBC_T4_665_MCAV,968899,857250,855406,855277,845889,187812
012024_BEL_CBC_T4_669_MCAV,1265565,1120659,1118086,1118485,1103353,253311


In [8]:
#exclude samples that are not in asv_clean
track_meta_no1 <- merge(meta, track_no1, by = "Tubelabel_species", all = FALSE)

## anova base r

In [10]:
class(track_meta_no1$Seq_run)

[1] "integer"

In [11]:
track_meta_no1$Seq_run <- as.character(track_meta_no1$Seq_run)

In [13]:
class(track_meta_no1$Seq_run)

[1] "character"

In [16]:
class(track_meta_no1$Month_year)

[1] "integer"

In [17]:
unique(track_meta_no1$Seq_run)

[1] "2" "3" "4" "5"

In [18]:
# Compute the analysis of variance
nonchim.anov <- aov(nonchim ~ Date_16S + factor(Month_year) + as.factor(Seq_run) + Species + Health_status, data = track_meta_no1)
# Summary of the analysis
summary(nonchim.anov)

                    Df    Sum Sq   Mean Sq F value   Pr(>F)    
Date_16S            18 5.057e+12 2.809e+11   6.580 4.25e-14 ***
factor(Month_year)  11 1.628e+11 1.480e+10   0.347    0.974    
as.factor(Seq_run)   1 9.921e+10 9.921e+10   2.324    0.128    
Species              5 2.456e+12 4.912e+11  11.504 2.83e-10 ***
Health_status        3 1.489e+11 4.963e+10   1.162    0.324    
Residuals          339 1.447e+13 4.270e+10                     
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

In [22]:
# Compute the analysis of variance
nodate.anov <- aov(nonchim ~ factor(Month_year) + as.factor(Seq_run) + Species + Health_status, data = track_meta_no1)
# Summary of the analysis
summary(nodate.anov)

                    Df    Sum Sq   Mean Sq F value  Pr(>F)    
factor(Month_year)  11 3.806e+11 3.460e+10   0.683 0.75499    
as.factor(Seq_run)   3 6.275e+11 2.092e+11   4.128 0.00676 ** 
Species              5 3.185e+12 6.369e+11  12.569   3e-11 ***
Health_status        3 2.144e+11 7.147e+10   1.410 0.23946    
Residuals          355 1.799e+13 5.068e+10                    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

In [19]:
table(track_meta_no1$Seq_run)
table(track_meta_no1$Seq_run[complete.cases(track_meta_no1[, c("nonchim","Date_16S","Month_year","Seq_run","Species","Health_status")])])


 2  3  4  5 
95 93 95 95 


 2  3  4  5 
95 93 95 95 

In [20]:
table(track_meta_no1$Seq_run, track_meta_no1$Date_16S)

   
    1_7_2026 1_8_2026 1_9_2026 10_14_2025 12_3_2025 12_4_2025 2_10_2026
  2        0        0        0         16         7        13         0
  3       30       30       32          0         0         1         0
  4        0        0        0          0         0         0        33
  5        0        0        0          0         0         0         0
   
    2_11_2026 2_9_2026 3_10_2026 3_11_2026 3_17_2026 3_9_2026 8_11_2025
  2         0        0         0         0         0        0         7
  3         0        0         0         0         0        0         0
  4        31       31         0         0         0        0         0
  5         0        0        31        31         2       31         0
   
    8_22_2025 8_26_2025 8_28_2025 8_8_2025 9_10_2025
  2         7         8        15        7        15
  3         0         0         0        0         0
  4         0         0         0        0         0
  5         0         0         0        0         0

In [21]:
# Check levels actually used
model.frame(nonchim.anov)$Seq_run |> table()

# Check missing data impact
sum(!complete.cases(track_meta_no1[, c("nonchim","Date_16S","Month_year","Seq_run","Species","Health_status")]))

# Check confounding
xtabs(~ Seq_run + Date_16S, data = track_meta_no1)

< table of extent 0 >

[1] 0

       Date_16S
Seq_run 1_7_2026 1_8_2026 1_9_2026 10_14_2025 12_3_2025 12_4_2025 2_10_2026
      2        0        0        0         16         7        13         0
      3       30       30       32          0         0         1         0
      4        0        0        0          0         0         0        33
      5        0        0        0          0         0         0         0
       Date_16S
Seq_run 2_11_2026 2_9_2026 3_10_2026 3_11_2026 3_17_2026 3_9_2026 8_11_2025
      2         0        0         0         0         0        0         7
      3         0        0         0         0         0        0         0
      4        31       31         0         0         0        0         0
      5         0        0        31        31         2       31         0
       Date_16S
Seq_run 8_22_2025 8_26_2025 8_28_2025 8_8_2025 9_10_2025
      2         7         8        15        7        15
      3         0         0         0        0         0
      4         0    

## anova vegan

# consolidate dada2 runs 1-5, with rarefaction curves

## notes
- Seq run r1 does not isolate specific species, and dates 2_25_2025 and 2_26_2025 do not isolate specific concentrations
- seq run has super high reads compared to other 4 seq runs, perhaps due to pooling samples at 30ng and THEN selecting for a specific band. cannot verify how many ng of each sample were ultimately in the pool

## all files together



## rareify (way to minimize towards samples that have more reads, look at read coverage instead of total reads)
### to remove some of the coverage from the really big seqs


In [ ]:
## rarefaction curve

In [ ]:
## now run anova to see what is primary driver after rarefaction of samples

In [ ]:
## rerun just seq run 1 and see if it matches up with original data attempt